# Download Sentinel2 training imagery
* Download low-tide cloud free satellite iamges closest to the UAV image collection

In [ ]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

# Values to edit

In [ ]:
max_cloud_covers = [5] #[1, 5, 10, 15] # percentage
tide_deltas = {1: {"hrs": 1, "mins": 00}} # {1: {"hrs": 1, "mins": 00},  2: {"hrs": 0, "mins": 45}, 3: {"hrs": 0, "mins": 30}}
lowtide_search_range = 60 # days - +/- this devided by 2

In [ ]:
site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"

In [ ]:
all_baselines = sentinel2.load_band_quality_baseline()

for tide_delta in tide_deltas.values():
    for max_cloud_cover in max_cloud_covers:
        print(f"\n\nMax cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        
        for site_name in site_names:
            site_baseline = all_baselines.loc[all_baselines["site"] == site_name]
            for i in range(3):
                try:
                    sampling.get_site_satellite(
                        site_name=site_name,
                        survey_dates_file=servey_dates_file,
                        lowtide_search_range=lowtide_search_range,
                        low_tide_delta_hrs=tide_delta["hrs"],
                        low_tide_delta_mins=tide_delta["mins"],
                        uav_folder=uav_folder,
                        max_cloud_cover=max_cloud_cover,
                        anomaly_filter_baseline=site_baseline,
                        n_mad=3,
                    )
                    break
                except Exception:
                    print("Caught exception downloading images and try again.")
            print(f"Save RGB for {site_name}")
            satellite_images_path = utils.get_training_satellite_path(site_name=site_name,  low_tide_delta_hrs=tide_delta["hrs"],
                                                             low_tide_delta_mins=tide_delta["mins"], max_cloud_cover=max_cloud_cover,
                                                             filtered=True)
            utils.rgb_from_satellite(satellite_data_path=satellite_images_path)